In [25]:
import pandas as pd

In [26]:
df = pd.read_csv('IPL.csv')

/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3524: DtypeWarning: Columns (28,29,30,31,43,46,47,48,51) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [27]:
df = df[['batter', 'batting_team', 'bowling_team', 'bowler', 'fielders']]
df

,batter,batting_team,bowling_team,bowler,fielders
0,SC Ganguly,Kolkata Knight Riders,Royal Challengers Bangalore,P Kumar,NaN
1,BB McCullum,Kolkata Knight Riders,Royal Challengers Bangalore,P Kumar,NaN
2,BB McCullum,Kolkata Knight Riders,Royal Challengers Bangalore,P Kumar,NaN
3,BB McCullum,Kolkata Knight Riders,Royal Challengers Bangalore,P Kumar,NaN
4,BB McCullum,Kolkata Knight Riders,Royal Challengers Bangalore,P Kumar,NaN
...,...,...,...,...,...
278200,Shashank Singh,Punjab Kings,Royal Challengers Bengaluru,JR Hazlewood,NaN
278201,Shashank Singh,Punjab Kings,Royal Challengers Bengaluru,JR Hazlewood,NaN
278202,Shashank Singh,Punjab Kings,Royal Challengers Bengaluru,JR Hazlewood,NaN
278203,Shashank Singh,Punjab Kings,Royal Challengers Bengaluru,JR Hazlewood,NaN


In [34]:
# Split and explode each role separately before concatenating
batters = df[['batter', 'batting_team']].rename(columns={'batter': 'player', 'batting_team': 'team'})

bowlers = df[['bowler', 'bowling_team']].rename(columns={'bowler': 'player', 'bowling_team': 'team'})

fielders = df[['fielders', 'bowling_team']].rename(columns={'fielders': 'player', 'bowling_team': 'team'})
fielders['player'] = fielders['player'].str.split(',')
fielders = fielders.explode('player')
fielders['player'] = fielders['player'].str.strip()

player_teams = pd.concat([batters, bowlers, fielders]).drop_duplicates().dropna().reset_index(drop=True)

In [35]:
player_teams[player_teams['player'].str.contains('Gayle', case=False)]

,player,team
165,CH Gayle,Kolkata Knight Riders
356,CH Gayle,Royal Challengers Bangalore
841,CH Gayle,Kings XI Punjab
984,CH Gayle,Punjab Kings


In [36]:
json_df = pd.read_csv('team_players.csv')

In [44]:
international_teams = [
    'India', 'Australia', 'England', 'South Africa', 'New Zealand',
    'Pakistan', 'Sri Lanka', 'West Indies', 'Bangladesh', 'Zimbabwe',
    'Afghanistan', 'Ireland', 'Scotland', 'Netherlands', 'UAE',
    'Namibia', 'Nepal', 'Oman', 'Papua New Guinea', 'USA'
]

countries_df = json_df[json_df['team'].isin(international_teams)]

combined = player_teams.merge(countries_df, on='player', how='left').rename(columns={'team_x': 'ipl_team', 'team_y': 'country'})

In [46]:
combined['country'] = combined['country'].fillna('India')

In [58]:
combined['ipl_team'] = combined['ipl_team'].replace({
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Delhi Daredevils': 'Delhi Capitals',
    'Royal Challengers Bengaluru': 'Royal Challengers Bangalore',
    'Pune Warriors': 'Rising Pune Supergiants',
    'Kings XI Punjab': 'Punjab Kings'
})
combined = combined.drop_duplicates().reset_index(drop=True)

In [61]:
combined.to_csv('player-country-team-dataset.csv', index=False)